In [3]:
import sys
from pathlib import Path

# Make src/ importable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from features import (
    load_cost_reports,
    standardise_columns,
    impute_income_columns,
    engineer_features,
    get_numerical_cols,
    get_categorical_cols,
    build_feature_set,
)

print("src/features.py loaded successfully")

ImportError: cannot import name 'load_cost_reports' from 'features' (/Users/amitkommineni/Documents/GitHub/Nursing-Home-Investment-Project/src/features.py)

In [ ]:
import os

# ── Portable path resolution ───────────────────────────────────────────────
# Set DATA_DIR env variable to your local data folder, or place CSVs in data/
# e.g. export DATA_DIR=/path/to/NH_Data
DATA_DIR = os.environ.get('DATA_DIR', 'data')
print(f'Loading data from: {DATA_DIR}')


In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load dataset individually
data_2015 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2015_CostReport.csv")
data_2016 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2016_CostReport.csv")
data_2017 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2017_CostReport.csv")
data_2018 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2018_CostReport.csv")
data_2019 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2019_CostReport.csv")
data_2020 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2020_CostReport.csv")
data_2021 = pd.read_csv(r"os.environ.get('DATA_DIR', 'data')\2021_CostReport.csv")

In [ ]:
data_2015['Year'] = '2015'
data_2016['Year'] = '2016'
data_2017['Year'] = '2017'
data_2018['Year'] = '2018'
data_2019['Year'] = '2019'
data_2020['Year'] = '2020'
data_2021['Year'] = '2021'

In [ ]:
def normalize_column_names(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

In [ ]:
# Apply normalization to all datasets
datasets = [
    normalize_column_names(data_2015),
    normalize_column_names(data_2016),
    normalize_column_names(data_2017),
    normalize_column_names(data_2018),
    normalize_column_names(data_2019),
    normalize_column_names(data_2020),
    normalize_column_names(data_2021)
]

# Determine common columns
common_columns = set(datasets[0].columns)
for df in datasets[1:]:
    common_columns.intersection_update(df.columns)

# Convert set to list for indexing
common_columns = list(common_columns)

# Merge all datasets using only the common columns
merged_data = pd.concat([df[common_columns] for df in datasets], ignore_index=True)

# Optional: Display the head of the merged dataset to confirm
print(merged_data.head())

total_assets zip_code  total_discharges_title_other  year state_code  \
0      765020.0    24244                           9.0  2015         VA   
1     1687456.0     6360                          36.0  2015         CT   
2      244239.0    50144                           3.0  2015         IA   
3     1830758.0     2132                          13.0  2015         MA   
4      807209.0    65802                           4.0  2015         MO   

   type_of_control                         facility_name  \
0              4.0  RIDGECREST MANOR NURSING & REHAB CTR   
1              4.0        REGENCY HEIGHTS OF NORWICH LLC   
2              4.0            WESTVIEW ACRES CARE CENTER   
3              4.0          VERO HEALTH & REHAB  PARKWAY   
4              4.0         WILSONS CREEK NURSING & REHAB   

   snf_discharges_title_xviii  rpt_rec_num  gross_revenue  ...  \
0                         5.0      1089712      1800296.0  ...   
1                        23.0      1091410      2843541.0  ...   
2                         3.0      1093283       613243.0  ...   
3                         6.0      1095547      1935277.0  ...   
4                         NaN      1095966       818175.0  ...   

   cash_on_hand_and_in_banks  snf_discharges_title_xix  rural_versus_urban  \
0                    54091.0                      35.0                   U   
1                  -339775.0                       NaN                   U   
2                    87279.0                       3.0                   R   
3                   352766.0                      13.0                   U   
4                    49316.0                      24.0                   U   

   provider_ccn  snf_bed_days_available  total_current_liabilities  \
0        495134                  7320.0                   453591.0   
1         75417                  8550.0                  2765768.0   
2        165252                  5520.0                   192098.0   
3        225497                  8601.0                  1824376.0   
4        265161                  5332.0                   808822.0   

   snf_days_title_xix  total_rug_days  total_current_assets  \
0              5853.0           333.0             1439227.0   
1                 NaN           856.0              980150.0   
2              2548.0            71.0              244239.0   
3              2881.0           349.0             1728824.0   
4              3945.0           111.0              807209.0   

   total_other_assets  
0           -729268.0  
1             36000.0  
2                 NaN  
3             74985.0  
4                 NaN  

[5 rows x 64 columns]

In [ ]:
merged_data = merged_data.drop(columns=['fiscal_year_begin_date','fiscal_year_end_date'])

In [ ]:
investment =  2000000  # $2 million
 
merged_data['ROI'] = merged_data['net_income'] / investment
merged_data['Operating_Margin'] = merged_data['net_income'] / merged_data['gross_revenue']
merged_data['Debt_to_equity_ratio'] = merged_data['total_liabilities'] / merged_data['total_fund_balances']
merged_data['Current_Ratio'] = merged_data['total_current_assets'] / merged_data['total_current_liabilities']
merged_data['Days_in_Accounts_Receivable'] = (merged_data['accounts_receivable'] / merged_data['net_patient_revenue']) * 365
merged_data['Bed_Occupancy_Rate'] = (merged_data['total_days_total'] / (merged_data['number_of_beds'] * 365)) * 100

# Thresholds for the ratios
ROI_Threshold = 0.05
OpMargin_Threshold = 0.02
DTE_Threshold = 1.5
CR_Threshold = 1.5
DAR_Threshold = 80
BOR_Threshold = 0.8

In [ ]:
def calculate_investment_choice(df):

    condition1 = ((df['ROI'] > ROI_Threshold) | (df['Operating_Margin'] > OpMargin_Threshold))
    condition2 = ((df['Debt_to_equity_ratio'] < DTE_Threshold) | (df['Current_Ratio'] >CR_Threshold))
    condition3 = ((df['Days_in_Accounts_Receivable'] < DAR_Threshold) | (df['Bed_Occupancy_Rate'] > BOR_Threshold))

    # Combine all conditions using logical AND
    final_condition = condition1 & condition2 & condition3

    # Apply the final condition to create the 'Investment_Choice' column
    df['Investment_Choice'] = np.where(final_condition, 1, 0)
    return df

In [ ]:
# Calculate investment choice
merged_data = calculate_investment_choice(merged_data)

# Print the first few rows to verify the new column
print(merged_data[['ROI', 'Operating_Margin', 'Debt_to_equity_ratio', 
                   'Current_Ratio', 'Days_in_Accounts_Receivable', 
                   'Bed_Occupancy_Rate', 'Investment_Choice']].head())

ROI  Operating_Margin  Debt_to_equity_ratio  Current_Ratio  \
0  0.155715          0.172988              1.456475       3.172962   
1 -0.294235         -0.206949             -1.344593       0.354386   
2 -0.023929         -0.078042              3.684202       1.271429   
3 -0.031819         -0.032883            285.862739       0.947625   
4 -0.000807         -0.001971           -501.439554       0.998006   

   Days_in_Accounts_Receivable  Bed_Occupancy_Rate  Investment_Choice  
0                   328.094346           15.342466                  1  
1                   200.315842           19.459265                  0  
2                    82.643013           16.324201                  0  
3                   323.880118           11.877975                  0  
4                   301.027303            7.540618                  0

In [ ]:
# Calculate the number of missing values per column
missing_values = merged_data.isnull().sum()
print(missing_values[missing_values > 0])

total_assets                    2779
total_discharges_title_other    3734
type_of_control                   46
snf_discharges_title_xviii      2700
gross_revenue                   2283
                                ... 
Operating_Margin                2570
Debt_to_equity_ratio            3268
Current_Ratio                   3127
Days_in_Accounts_Receivable     5041
Bed_Occupancy_Rate              2410
Length: 61, dtype: int64

In [ ]:
# Visualizing missing values using a heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(merged_data.isnull(), cbar=False)
plt.title('Heatmap of Missing Values in Dataset')
plt.show()

In [ ]:
# Get list of numerical columns
numerical_cols = merged_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Get list of categorical columns
categorical_cols = merged_data.select_dtypes(include=['object', 'category']).columns.tolist()

# Print the lists
print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

Numerical Columns: ['total_assets', 'total_discharges_title_other', 'type_of_control', 'snf_discharges_title_xviii', 'rpt_rec_num', 'gross_revenue', 'total_days_total', 'net_income', 'snf_days_title_xviii', 'net_patient_revenue', 'less_total_operating_expense', 'snf_discharges_title_other', 'total_discharges_title_xix', 'other_assets', 'fixed_equipment', 'buildings', 'total_days_other', 'snf_admissions_other', 'general_fund_balance', 'snf_days_other', 'snf_number_of_beds', 'other_current_liabilities', 'inpatient_pps_amount', 'snf_admissions_title_xviii', 'total_costs', 'inpatient_revenue', 'total_fixed_assets', 'major_movable_equipment', 'total_days_title_xviii', 'total_bed_days_available', 'snf_admissions_total', 'snf_days_total', 'total_salaries_from_worksheet_a', 'total_liabilities', 'total_discharges_title_xviii', 'snf_admissions_title_xix', 'total_days_title_xix', 'total_discharges_total', 'accounts_payable', 'snf_discharges_total', 'accounts_receivable', 'total_income', 'total_fund_balances', 'number_of_beds', 'cash_on_hand_and_in_banks', 'snf_discharges_title_xix', 'provider_ccn', 'snf_bed_days_available', 'total_current_liabilities', 'snf_days_title_xix', 'total_rug_days', 'total_current_assets', 'total_other_assets', 'ROI', 'Operating_Margin', 'Debt_to_equity_ratio', 'Current_Ratio', 'Days_in_Accounts_Receivable', 'Bed_Occupancy_Rate']
Categorical Columns: ['zip_code', 'year', 'state_code', 'facility_name', 'county', 'street_address', 'city', 'medicare_cbsa_number', 'rural_versus_urban']

total_assets                    float64
zip_code                         object
total_discharges_title_other    float64
year                             object
state_code                       object
                                 ...   
Debt_to_equity_ratio            float64
Current_Ratio                   float64
Days_in_Accounts_Receivable     float64
Bed_Occupancy_Rate              float64
Investment_Choice                 int32
Length: 69, dtype: object

In [ ]:
# Perform mean imputation on numerical columns
merged_data.fillna({col: merged_data[col].mean() for col in numerical_cols}, inplace=True)


# Perform mode imputation on categorical columns
for col in categorical_cols:
    mode_value = merged_data[col].mode()[0]  # mode() returns a Series, take the first element
    merged_data[col] = merged_data[col].fillna(mode_value)

In [ ]:
merged_data.isnull().sum()

total_assets                    0
zip_code                        0
total_discharges_title_other    0
year                            0
state_code                      0
                               ..
Debt_to_equity_ratio            0
Current_Ratio                   0
Days_in_Accounts_Receivable     0
Bed_Occupancy_Rate              0
Investment_Choice               0
Length: 69, dtype: int64

In [ ]:
# Save the DataFrame to CSV in the current working directory
merged_data.to_csv('Investment_Choice_Data.csv', index=False)

print("Data saved to 'Investment_Choice_Data.csv' in the current working directory.")

Data saved to 'Investment_Choice_Data.csv' in the current working directory.

In [ ]:
# Count of unique values per categorical column
for column in merged_data.select_dtypes(include=['object', 'category']).columns:
    print(f"Unique values in {column}: {merged_data[column].nunique()}")

Unique values in zip_code: 12075
Unique values in year: 7
Unique values in state_code: 52
Unique values in facility_name: 23079
Unique values in county: 2203
Unique values in street_address: 19752
Unique values in city: 5644
Unique values in medicare_cbsa_number: 1111
Unique values in rural_versus_urban: 2

In [ ]:
# Count the number of entries for each Investment Choice
investment_counts = merged_data['Investment_Choice'].value_counts()

# Create a pie chart
plt.figure(figsize=(6,6))  # Set the size of the figure
plt.pie(investment_counts, labels=investment_counts.index, autopct='%1.1f%%', startangle=140, 
        colors=['brown', 'green'])
plt.title('Distribution of Investment Choices')
plt.show()

In [ ]:
# Identify numerical columns
numerical_data = merged_data.select_dtypes(include=['int64', 'float64'])

# Calculate the number of subplots needed
num_plots = numerical_data.shape[1]
num_rows = int(np.ceil(num_plots / 5))  # Setting up 5 columns per row

# Create histogram plots
numerical_data.hist(bins=15, figsize=(15, num_rows * 3), layout=(num_rows, 5))
plt.tight_layout()
plt.show()

In [ ]:
ratios = ['ROI', 'Operating_Margin', 'Debt_to_equity_ratio', 'Current_Ratio', 'Bed_Occupancy_Rate']
target = 'Investment_Choice'

# Display basic statistics for the ratios
print(merged_data[ratios].describe())

ROI  Operating_Margin  Debt_to_equity_ratio  Current_Ratio  \
count  106269.000000     106269.000000          1.062690e+05  106269.000000   
mean        0.034650         -0.146422         -6.281096e+02      10.074346   
std         1.212074         40.482346          2.094278e+05    1856.663708   
min       -83.734805     -13159.085366         -6.194572e+07 -117366.250000   
25%        -0.186759         -0.053693         -1.922486e+00       0.686134   
50%         0.034650          0.005998          2.445886e-01       1.314284   
75%         0.269234          0.056845          1.505992e+00       2.586726   
max       104.800237        549.046625          1.095649e+07  554139.333333   

       Bed_Occupancy_Rate  
count       106269.000000  
mean            75.200054  
std            110.679494  
min              0.000894  
25%             64.535769  
50%             78.907534  
75%             89.044319  
max          25371.033859

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(merged_data['ROI'], bins=500, color='skyblue', kde=True)
plt.title('Histogram of ROI')
plt.xlabel('ROI')
plt.ylabel('Frequency')
plt.xlim(-5, 5) 
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(merged_data['Operating_Margin'], bins=200, color='skyblue', kde=True)
plt.title('Histogram of Operating Margin')
plt.xlabel('Operating Margin')
plt.ylabel('Frequency')
#plt.xlim(-100,10)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(merged_data['Debt_to_equity_ratio'], bins=100, color='skyblue', kde=True)
plt.title('Histogram of Debt to Equity Ratio')
plt.xlabel('Debt to Equity Ratio')
plt.ylabel('Frequency')
#plt.xlim(-10,10)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(merged_data['Bed_Occupancy_Rate'], bins=800, color='skyblue', kde=True)
plt.title('Histogram of Bed Occupancy Rate')
plt.xlabel('Bed Occupancy Rate')
plt.ylabel('Frequency')
plt.xlim(-10,300)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(merged_data['Current_Ratio'], bins=5000, color='skyblue', kde=True)
plt.title('Histogram of Current Ratio')
plt.xlabel('Current Ratio')
plt.ylabel('Frequency')
plt.xlim(-500,500)
plt.show()

In [ ]:
#create the cross table
contingency_table = pd.crosstab(merged_data['Investment_Choice'], merged_data['rural_versus_urban'])

#display the cross table
print(contingency_table)

rural_versus_urban      R      U
Investment_Choice               
0                   16787  49221
1                   11519  28742

In [ ]:
# Calculate proportions
proportions = merged_data.groupby(['Investment_Choice', 'rural_versus_urban']).size().unstack(level=-1)
proportions = proportions.div(proportions.sum(axis=1), axis=0).reset_index()

# Melt the DataFrame to reshape it for plotting
proportions = proportions.melt(id_vars='Investment_Choice', var_name='rural_versus_urban', value_name='proportion')

# Create count plot
plt.figure(figsize=(8, 6))
sns.barplot(x='Investment_Choice', y='proportion', hue='rural_versus_urban', data=proportions)
plt.title('Investment choice by rural/urban')
plt.xlabel('Investment Choice')
plt.ylabel('Proportion')
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

#perform the chi-squared test
chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

#print the results
print("Chi-squared statistic", chi2_stat)
print("P-value:", p_val)
print(expected)

Chi-squared statistic 129.17205359884184
P-value: 6.218724354508406e-30
[[17582.00837497 48425.99162503]
 [10723.99162503 29537.00837497]]

These values suggest that there is a statistically significant relationship between 'Investment Choice' and 'Rural vs Urban'

In [ ]:
# Filter out the numerical columns
numerical_data = merged_data.select_dtypes(include=['int64', 'float64'])

# Calculate the correlation matrix
correlation_matrix = numerical_data.corr()

# Set up the matplotlib figure
plt.figure(figsize=(12, 8))

# Create a mask for the upper triangle
tri_matrix = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Set up the matplotlib figure
plt.figure(figsize=(10, 8))

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(correlation_matrix, cmap='coolwarm', linewidths=0.5, square=True,
            mask=tri_matrix, vmin=-1, vmax=1, cbar_kws={'shrink': .8})

# Add a title for clarity
plt.title('Correlation Matrix Heatmap')
plt.show()

<Figure size 1200x800 with 0 Axes>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = merged_data.drop('Investment_Choice', axis=1).drop(categorical_cols, axis=1)
y = merged_data['Investment_Choice'] 

# Step 1: Split the data into training (75%) and remaining data (25%)
X_train, X_remaining, y_train, y_remaining = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 2: Split the remaining data into test (60%) and validation (40%)
X_test, X_val, y_test, y_val = train_test_split(X_remaining, y_remaining, test_size=0.4, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

# Apply PCA
pca = PCA(n_components=40)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
X_val_pca = pca.transform(X_val_scaled)

# Check the size of each split to confirm the proportions
print("Training set size:", len(X_train_pca))
print("Testing set size:", len(X_test_pca))
print("Validation set size:", len(X_val_pca))

Training set size: 79701
Testing set size: 15940
Validation set size: 10628

In [ ]:
# Exploring the variance explained by each principal component
print("Explained variance by each component:", pca.explained_variance_ratio_)
print("Total variance explained:", sum(pca.explained_variance_ratio_))

Explained variance by each component: [0.1739274  0.12485483 0.07264234 0.06644517 0.05186707 0.04385522
 0.03959504 0.03608036 0.03389251 0.03291952 0.03065718 0.02581912
 0.02458199 0.02146508 0.01991952 0.01743738 0.01698193 0.01695364
 0.0169326  0.01662047 0.01619774 0.01391435 0.0118346  0.00985504
 0.00717685 0.00702162 0.0063276  0.00558224 0.00524651 0.00428581
 0.00416153 0.00358755 0.00323335 0.00302877 0.00293143 0.0025682
 0.00216588 0.00165731 0.00136952 0.00094905]
Total variance explained: 0.9965433016344509

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1, 41), np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance Ratio')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Initialize KNN classifier
knn = KNeighborsClassifier()

# Define a range of n_neighbors values to test
n_neighbors_values = [23, 25, 27, 29, 31]

# Perform cross-validation for each n_neighbors value
for n_neighbors in n_neighbors_values:
    knn.n_neighbors = n_neighbors
    
    # Cross-validation on training set
    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(f"Mean cross-validation accuracy for n_neighbors={n_neighbors}: {cv_scores.mean():.4f} (std: {cv_scores.std():.4f})")

In [ ]:
Mean cross-validation accuracy for n_neighbors=23: 0.7976 (std: 0.0026)
Mean cross-validation accuracy for n_neighbors=25: 0.7974 (std: 0.0020)
Mean cross-validation accuracy for n_neighbors=27: 0.7983 (std: 0.0016)
Mean cross-validation accuracy for n_neighbors=29: 0.7972 (std: 0.0016)
Mean cross-validation accuracy for n_neighbors=31: 0.7964 (std: 0.0012)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# Define the range of n_neighbors values to test
n_neighbors_values = [23, 25, 27, 29, 31]

# Initialize lists to store mean accuracies and standard deviations
mean_accuracies = []
std_accuracies = []

# Perform cross-validation for each n_neighbors value
for n_neighbors in n_neighbors_values:
    # Initialize KNN classifier with current n_neighbors value
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)
    
    # Perform cross-validation on training set
    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    
    # Calculate mean and standard deviation of cross-validation scores
    mean_accuracies.append(np.mean(cv_scores))
    std_accuracies.append(np.std(cv_scores))

# Plot mean cross-validation accuracy along with standard deviation
plt.errorbar(n_neighbors_values, mean_accuracies, yerr=std_accuracies, fmt='-o', color='b')
plt.title('Mean Cross-Validation Accuracy vs. n_neighbors')
plt.xlabel('n_neighbors')
plt.ylabel('Mean Accuracy')
plt.xticks(n_neighbors_values)
plt.grid(True)
plt.show()

In [ ]:
# Initialize KNN classifier with n_neighbors=27
knn = KNeighborsClassifier(n_neighbors=27)

# Perform cross-validation on training set
cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
print(f"Mean cross-validation accuracy for n_neighbors=27: {cv_scores.mean():.4f} (std: {cv_scores.std():.4f})")

# Train the model on the entire training set
knn.fit(X_train_scaled, y_train)

# Evaluate the model on the train set
train_pred = knn.predict(X_train_scaled)
train_accuracy_knn = accuracy_score(y_train, train_pred)
print(f"Train accuracy for n_neighbors=27: {train_accuracy_knn:.4f}")

# Evaluate the model on the test set
test_pred = knn.predict(X_test_scaled)
test_accuracy_knn = accuracy_score(y_test, test_pred)
print(f"Test accuracy for n_neighbors=27: {test_accuracy_knn:.4f}")

# Evaluate the model on the validation set
val_pred = knn.predict(X_val_scaled)
val_accuracy_knn = accuracy_score(y_val, val_pred)
print(f"Validation accuracy for n_neighbors=27: {val_accuracy_knn:.4f}")

In [ ]:
Mean cross-validation accuracy for n_neighbors=27: 0.7983 (std: 0.0016)
Train accuracy for n_neighbors=27: 0.8234
Test accuracy for n_neighbors=27: 0.8044
Validation accuracy for n_neighbors=27: 0.7942

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialize the Random Forest model
rf = RandomForestClassifier(n_estimators=100, 
                            max_features=5,
                            max_depth=4,
                            min_samples_split=100,
                            random_state=42) 

# Train the model using the training set
rf.fit(X_train_scaled, y_train)

# Predict and evaluate on the training set
y_train_pred = rf.predict(X_train_scaled)  
train_accuracy_rf = accuracy_score(y_train, y_train_pred)
print("Training set accuracy:", train_accuracy_rf)

# Predict and evaluate on the test set
y_test_pred = rf.predict(X_test_scaled) 
test_accuracy_rf = accuracy_score(y_test, y_test_pred)
print("Test set accuracy:", test_accuracy_rf)

# Predict and evaluate on the validation set
y_val_pred = rf.predict(X_val_scaled)  
val_accuracy_rf = accuracy_score(y_val, y_val_pred)
print("Validation set accuracy:", val_accuracy_rf)

Training set accuracy: 0.9193611121566856
Test set accuracy: 0.9210790464240903
Validation set accuracy: 0.913153933007151

In [ ]:
import matplotlib.pyplot as plt

# Visualize feature importance
feature_importance = rf.feature_importances_
sorted_idx = np.argsort(feature_importance)
plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(X_train.columns)[sorted_idx])
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.show()

In [ ]:
# Visualize a single decision tree from the Random Forest 
from sklearn.tree import plot_tree

# Choose a tree index to visualize (e.g., the first tree)
tree_index = 0
plt.figure(figsize=(20, 10))
plot_tree(rf.estimators_[tree_index], feature_names=X_train.columns, filled=True)
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize the Logistic Regression model with increased max_iter and a different solver
log_reg = LogisticRegression(max_iter=1000, solver='sag', random_state=42)

# Train the model using the training set
log_reg.fit(X_train_pca, y_train)

# Predict and evaluate on the training set
y_train_pred = log_reg.predict(X_train_pca)
train_accuracy_lr = accuracy_score(y_train, y_train_pred)
print("Training set accuracy:", train_accuracy_lr)

# Predict and evaluate on the test set
y_test_pred = log_reg.predict(X_test_pca)
test_accuracy_lr = accuracy_score(y_test, y_test_pred)
print("Test set accuracy:", test_accuracy_lr)

# Predict and evaluate on the validation set
y_val_pred = log_reg.predict(X_val_pca)
val_accuracy_lr = accuracy_score(y_val, y_val_pred)
print("Validation set accuracy:", val_accuracy_lr)

Training set accuracy: 0.8303534460044416
Test set accuracy: 0.8297365119196989
Validation set accuracy: 0.8193451260820475

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define models and their accuracies
models = ['KNN', 'Random Forest', 'Logistic Regression']
accuracy_train = [train_accuracy_knn, train_accuracy_rf, train_accuracy_lr]
accuracy_test = [test_accuracy_knn, test_accuracy_rf, test_accuracy_lr]
accuracy_val = [val_accuracy_knn, val_accuracy_rf, val_accuracy_lr]

# Plotting
fig, ax = plt.subplots(figsize=(8, 8))
index = np.arange(len(models))
bar_width = 0.25  
opacity = 0.8

rects1 = ax.bar(index, accuracy_train, bar_width, alpha=opacity, color='skyblue', label='Training Accuracy')
rects2 = ax.bar(index + bar_width, accuracy_test, bar_width, alpha=opacity, color='coral', label='Testing Accuracy')
rects3 = ax.bar(index + 2*bar_width, accuracy_val, bar_width, alpha=opacity, color='lightgreen', label='Validation Accuracy')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison by Training, Testing, and Validation Accuracy')
ax.set_xticks(index + bar_width)
ax.set_xticklabels(models)
ax.legend(loc='upper right')

#plt.tight_layout()
plt.show()

1) Accuracy
2) Precision, Recall, F1 Score
3) Confusion Matrix
4) ROC AUC Curve

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def evaluate_model(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    conf_mat = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Confusion Matrix:")
    print(conf_mat)

    if y_prob is not None:  # ROC AUC requires probability scores
        roc_auc = roc_auc_score(y_true, y_prob[:, 1])  # Assuming y_prob is a 2-column array with probabilities
        print(f"ROC AUC: {roc_auc:.4f}")
    
    return acc, prec, rec, f1, roc_auc if y_prob is not None else None

# KNeighborsClassifier
print("\nK-Nearest Neighbors Model Evaluation:")
y_train_prob_knn = knn.predict_proba(X_train_scaled)
y_test_prob_knn = knn.predict_proba(X_test_scaled)
y_val_prob_knn = knn.predict_proba(X_val_scaled)

print("\nTraining Set Evaluation:")

train_metrics_knn = evaluate_model(y_train, y_train_pred, y_train_prob_knn)

print("\nTest Set Evaluation:")
test_metrics_knn = evaluate_model(y_test, y_test_pred, y_test_prob_knn)

print("\nValidation Set Evaluation:")
val_metrics_knn = evaluate_model(y_val, y_val_pred, y_val_prob_knn)



# RandomForestClassifier
print("Random Forest Model Evaluation:")
y_train_prob_rf = rf.predict_proba(X_train_scaled)
y_test_prob_rf = rf.predict_proba(X_test_scaled)
y_val_prob_rf = rf.predict_proba(X_val_scaled)

print("\nTraining Set Evaluation:")
train_metrics_rf = evaluate_model(y_train, y_train_pred, y_train_prob_rf)

print("\nTest Set Evaluation:")
test_metrics_rf = evaluate_model(y_test, y_test_pred, y_test_prob_rf)

print("\nValidation Set Evaluation:")
val_metrics_rf = evaluate_model(y_val, y_val_pred, y_val_prob_rf)

# LogisticRegression
print("\nLogistic Regression Model Evaluation:")
y_train_prob_lr = log_reg.predict_proba(X_train_pca)
y_test_prob_lr = log_reg.predict_proba(X_test_pca)
y_val_prob_lr = log_reg.predict_proba(X_val_pca)

print("\nTraining Set Evaluation:")
train_metrics_lr = evaluate_model(y_train, y_train_pred, y_train_prob_lr)

print("\nTest Set Evaluation:")
test_metrics_lr = evaluate_model(y_test, y_test_pred, y_test_prob_lr)

print("\nValidation Set Evaluation:")
val_metrics_lr = evaluate_model(y_val, y_val_pred, y_val_prob_lr)

K-Nearest Neighbors Model Evaluation:

Training Set Evaluation:
Accuracy: 0.9194
Precision: 0.8345
Recall: 0.9824
F1 Score: 0.9024
Confusion Matrix:
[[43565  5894]
 [  533 29709]]
ROC AUC: 0.9053

Test Set Evaluation:
Accuracy: 0.9211
Precision: 0.8351
Recall: 0.9836
F1 Score: 0.9033
Confusion Matrix:
[[8806 1160]
 [  98 5876]]
ROC AUC: 0.8820

Validation Set Evaluation:
Accuracy: 0.9132
Precision: 0.8240
Recall: 0.9815
F1 Score: 0.8959
Confusion Matrix:
[[5735  848]
 [  75 3970]]
ROC AUC: 0.8757
Random Forest Model Evaluation:

Training Set Evaluation:
Accuracy: 0.9194
Precision: 0.8345
Recall: 0.9824
F1 Score: 0.9024
Confusion Matrix:
[[43565  5894]
 [  533 29709]]
ROC AUC: 0.9922

Test Set Evaluation:
Accuracy: 0.9211
Precision: 0.8351
Recall: 0.9836
F1 Score: 0.9033
Confusion Matrix:
[[8806 1160]
 [  98 5876]]
ROC AUC: 0.9928

Validation Set Evaluation:
Accuracy: 0.9132
Precision: 0.8240
Recall: 0.9815
F1 Score: 0.8959
Confusion Matrix:
[[5735  848]
 [  75 3970]]
ROC AUC: 0.9913

Logistic Regression Model Evaluation:

Training Set Evaluation:
Accuracy: 0.9194
Precision: 0.8345
Recall: 0.9824
F1 Score: 0.9024
Confusion Matrix:
[[43565  5894]
 [  533 29709]]
ROC AUC: 0.9258

Test Set Evaluation:
Accuracy: 0.9211
Precision: 0.8351
Recall: 0.9836
F1 Score: 0.9033
Confusion Matrix:
[[8806 1160]
 [  98 5876]]
ROC AUC: 0.9267

Validation Set Evaluation:
Accuracy: 0.9132
Precision: 0.8240
Recall: 0.9815
F1 Score: 0.8959
Confusion Matrix:
[[5735  848]
 [  75 3970]]
ROC AUC: 0.9217

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

def plot_roc_curve(y_true, y_prob, model_name):
    
    # Compute ROC curve and ROC area
    fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])  # y_prob[:,1]: probability estimates for the positive class
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic - {model_name}')
    plt.legend(loc="lower right")
    plt.show()

# Plot ROC Curve for K-Nearest Neighbors
print("ROC Curve for K-Nearest Neighbors:")
plot_roc_curve(y_test, y_test_prob_knn, "K-Nearest Neighbors")

# Plot ROC Curve for Random Forest
print("ROC Curve for Random Forest:")
plot_roc_curve(y_test, y_test_prob_rf, "Random Forest")

# Plot ROC Curve for Logistic Regression
print("ROC Curve for Logistic Regression:")
plot_roc_curve(y_test, y_test_prob_lr, "Logistic Regression")

In [ ]:
ROC Curve for K-Nearest Neighbors:

In [ ]:
ROC Curve for Random Forest:

In [ ]:
ROC Curve for Logistic Regression: